# Tag suggester (workflow aid)

Suggests tags for a new case from its summary (multi-label one-vs-rest logistic regression). **Important:** tags are *leakage* — they are written after an agent understands the case and encode the target fields — so no predictive model (M1–M5) uses tags as a feature. These suggestions are shown to the agent only, to speed up and standardise tagging; they are never fed back into the models.

In [ ]:
import sys, os, warnings; warnings.filterwarnings("ignore")
search = os.getcwd()
while search != os.path.dirname(search) and not os.path.exists(os.path.join(search, "case_pipeline.py")):
    search = os.path.dirname(search)
sys.path.insert(0, search)
import numpy as np, pandas as pd
import case_pipeline as P   # shared loading / splits / features / evaluation

In [ ]:
df = P.load_clean()

# Train with tags as the LABEL and the summary as the input; keep tags seen >= 10 times.
suggester = P.fit_tag_suggester(df, text_col="case_summary", tag_col="tags", min_support=10)
print("modelled tags:", len(suggester["classes"]))

# Sanity check: suggestions for an example summary.
P.suggest_tags(suggester, "cannot export the monthly VAT invoice to Xero, 500 error", k=4)

In [ ]:
P.save_bundle("tag_suggester", {
    "vectorizer": suggester["vectorizer"], "model": suggester["model"],
    "classes": suggester["classes"], "min_support": suggester["min_support"],
    "kind": "tagger", "label": "Tag suggester",
    "meta": {"approach": "multi-label OvR logistic regression on the summary",
             "n_tags": len(suggester["classes"]),
             "note": "SUGGESTIONS ONLY — tags are leakage and are never used as model features"}})